# EMS 품질·Coverage EDA

**목적:** 모델 feature 설계 전에 DB 기준 분석 입력의 범위, 결측, 품질 counter, 시간 격자, 부호 규약을 먼저 확인한다.

이 노트북은 EMS 데이터 계약 기준의 입력 신뢰성을 점검하기 위한 1단계 EDA이다.

## 01. 실행 가정 및 제외 범위

### 가정

1. 기준 원천은 PostgreSQL/TimescaleDB `ems` schema이다.
2. 기본 mart는 `ems.cr_measurement_15min`, 보고·장기 추세 mart는 `ems.cr_measurement_1h`이다.
3. 파일 lineage와 품질 counter는 `ems.full_source_file`을 기준으로 확인한다.
4. `1min`은 `full_source_file` metadata의 원천/적재 이력 확인 범위로 둔다.
5. 본 노트북은 read-only 조회만 수행한다.

### 이번 단계에서 확인할 항목

1. DB relation 존재 여부와 row estimate
2. `cr_measurement_1min` DB mart 부재 여부
3. source file status, resolution, processing level 분포
4. 품질 counter 요약
5. meter·measurement coverage
6. EDA 목적별 대표 meter의 시간 격자 gap smoke check
7. EDA 목적별 대표 consumption/production meter의 부호 분포 smoke check

### 제외 범위

1. feature set 확정
2. 모델 학습
3. 이상탐지 threshold 산정
4. DB write 또는 schema 변경
5. 전체 계량기 상세 시각화
6. 팀원 상관관계 분석의 11개 전기 계량기별 correlation/feature 제거 판단 재수행

### 성공 기준

1. 노트북이 프로젝트 root 기준으로 실행 가능하다.
2. 기본 실행은 노트북 출력 중심으로 수행한다.
3. `SAVE_OUTPUTS = True` 설정 시에만 `outputs/tables/quality_coverage/`, `outputs/figures/quality_coverage/`에 검토용 산출물을 저장한다.
4. query 실패 시 error table을 표시한다.

### 계량기 선택 기준

본 노트북의 기본 coverage 분석은 81개 전체 meter와 전체 measurement metadata를 대상으로 수행한다. 대표 meter는 다음 EDA 축의 DB 조회 가능성을 확인하는 smoke check 용도이다.

1. `domain`: electricity, thermal, weather가 모두 포함되어야 한다.
2. `role`: consumption, production, thermal_flow, weather가 모두 포함되어야 한다.
3. `equipment_group`: 후속 EDA의 핵심 축인 central_cooling, server_power, chp, pv, thermal, weather를 포함한다.
4. `coupling`: 전기-열-기상 연결 가능성이 있는 계량기를 우선한다.
5. `sign convention`: consumption 양수, production 음수, W/W_in/W_out 방향성을 확인할 수 있어야 한다.
6. `data readiness`: `15min`과 `1h` mart에서 대표 measurement가 존재해야 한다.
7. `team analysis`: 팀원 상관관계 분석은 후속 해석의 참고 근거로 사용한다.

In [ ]:
# C01. 환경 설정 및 경로
from pathlib import Path
from datetime import datetime, timezone
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    from IPython.display import display
except ImportError:
    display = print

sns.set_theme(style="whitegrid")
plt.rcParams["axes.unicode_minus"] = False

ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").exists():
    for candidate in [ROOT, *ROOT.parents]:
        if (candidate / "pyproject.toml").exists() and candidate.name == "EMS":
            ROOT = candidate
            break

os.chdir(ROOT)
SAVE_OUTPUTS = False
OUT_TABLE = ROOT / "outputs" / "tables" / "quality_coverage"
OUT_FIG = ROOT / "outputs" / "figures" / "quality_coverage"
if SAVE_OUTPUTS:
    OUT_TABLE.mkdir(parents=True, exist_ok=True)
    OUT_FIG.mkdir(parents=True, exist_ok=True)

print(f"ROOT={ROOT}")
print(f"SAVE_OUTPUTS={SAVE_OUTPUTS}")
print(f"OUT_TABLE={OUT_TABLE}")
print(f"OUT_FIG={OUT_FIG}")
print(f"executed_at_utc={datetime.now(timezone.utc).isoformat()}")

In [ ]:
# C02. DB 접속 helper
import psycopg


def load_dotenv(path: Path) -> None:
    if not path.exists():
        return
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))


load_dotenv(ROOT / ".env")
REQUIRED_ENV = ["DB_HOST", "DB_PORT", "DB_NAME", "DB_USER", "DB_PASSWORD"]
missing_env = [key for key in REQUIRED_ENV if not os.environ.get(key)]
if missing_env:
    raise RuntimeError(f"DB 환경변수가 없습니다: {missing_env}")


def connect():
    return psycopg.connect(
        host=os.environ["DB_HOST"],
        port=os.environ["DB_PORT"],
        dbname=os.environ["DB_NAME"],
        user=os.environ["DB_USER"],
        password=os.environ["DB_PASSWORD"],
        connect_timeout=8,
    )


def query_df(sql: str, params=None, timeout_s: int = 20) -> pd.DataFrame:
    try:
        with connect() as conn:
            with conn.cursor() as cur:
                cur.execute(f"SET statement_timeout = '{int(timeout_s)}s'")
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", UserWarning)
                return pd.read_sql(sql, conn, params=params)
    except Exception as exc:
        return pd.DataFrame({
            "error_type": [type(exc).__name__],
            "error_message": [str(exc)],
            "sql_preview": [" ".join(sql.split())[:300]],
        })


def show_table(df: pd.DataFrame, name: str) -> None:
    rows = len(df)
    if SAVE_OUTPUTS:
        OUT_TABLE.mkdir(parents=True, exist_ok=True)
        out = OUT_TABLE / name
        df.to_csv(out, index=False)
        print(f"saved={out} rows={rows}")
    else:
        print(f"table={name} rows={rows}")
    if rows == 0:
        display(pd.DataFrame({"table": [name], "rows": [0], "status": ["empty"]}))
    else:
        display(df)


masked = {key: ("<set>" if key != "DB_PASSWORD" else "<masked>") for key in REQUIRED_ENV}
print(masked)

In [ ]:
# C03. EMS relation inventory
relation_df = query_df("""
WITH rel(relation_name) AS (
    VALUES
        ('full_source_file'),
        ('full_meter'),
        ('full_measurement_definition'),
        ('cr_measurement_15min'),
        ('cr_measurement_1h'),
        ('cr_measurement_1min')
), catalog AS (
    SELECT
        relation_name,
        to_regclass('ems.' || quote_ident(relation_name)) AS rel_oid
    FROM rel
)
SELECT
    c.relation_name AS relation,
    c.rel_oid IS NOT NULL AS exists,
    COALESCE(pg.reltuples::bigint, 0) AS row_estimate,
    pg_size_pretty(pg_total_relation_size(c.rel_oid)) AS total_size
FROM catalog c
LEFT JOIN pg_class pg ON pg.oid = c.rel_oid
ORDER BY c.relation_name
""")

show_table(relation_df, "relation_inventory.csv")

In [ ]:
# C04. CR mart timestamp range smoke check
mart_range_df = query_df("""
SELECT 'cr_measurement_1h' AS relation, min(ts) AS min_ts, max(ts) AS max_ts
FROM ems.cr_measurement_1h
UNION ALL
SELECT 'cr_measurement_15min' AS relation, min(ts) AS min_ts, max(ts) AS max_ts
FROM ems.cr_measurement_15min
ORDER BY relation
""", timeout_s=20)

show_table(mart_range_df, "mart_timestamp_range.csv")

In [ ]:
# C05. Source file status 및 row counter 요약
source_status_df = query_df("""
SELECT
    processing_level,
    resolution_code,
    status,
    count(*) AS source_files,
    sum(csv_rows) AS csv_rows,
    sum(inserted_rows) AS inserted_rows,
    sum(null_value_rows) AS null_value_rows,
    sum(invalid_value_rows) AS invalid_value_rows,
    sum(invalid_ts_rows) AS invalid_ts_rows,
    sum(non_finite_rows) AS non_finite_rows,
    sum(conflict_rows) AS conflict_rows,
    sum(duplicate_key_rows) AS duplicate_key_rows
FROM ems.full_source_file
GROUP BY processing_level, resolution_code, status
ORDER BY processing_level, resolution_code, status
""")

show_table(source_status_df, "source_status_summary.csv")

In [ ]:
# C06. 품질 counter 비율 요약
quality_cols = [
    "null_value_rows",
    "invalid_value_rows",
    "invalid_ts_rows",
    "non_finite_rows",
    "conflict_rows",
    "duplicate_key_rows",
]

quality_summary_df = source_status_df.copy()
if "error_type" not in quality_summary_df.columns and not quality_summary_df.empty:
    for col in quality_cols:
        quality_summary_df[f"{col}_rate"] = np.where(
            quality_summary_df["csv_rows"].fillna(0).to_numpy() > 0,
            quality_summary_df[col].fillna(0) / quality_summary_df["csv_rows"].fillna(0),
            np.nan,
        )

show_table(quality_summary_df, "quality_counter_rates.csv")

In [ ]:
# C07. Meter·measurement coverage inventory
coverage_df = query_df("""
SELECT
    processing_level,
    resolution_code,
    meter_urn,
    measurement,
    status,
    count(*) AS source_files,
    sum(csv_rows) AS csv_rows,
    sum(inserted_rows) AS inserted_rows,
    sum(null_value_rows) AS null_value_rows,
    sum(invalid_value_rows) AS invalid_value_rows,
    sum(non_finite_rows) AS non_finite_rows
FROM ems.full_source_file
GROUP BY processing_level, resolution_code, meter_urn, measurement, status
ORDER BY processing_level, resolution_code, meter_urn, measurement, status
""", timeout_s=20)

show_table(coverage_df, "meter_measurement_coverage.csv")

if "error_type" not in coverage_df.columns and not coverage_df.empty:
    coverage_summary = (
        coverage_df[coverage_df["status"].eq("loaded")]
        .groupby(["processing_level", "resolution_code"], dropna=False)
        .agg(
            meters=("meter_urn", "nunique"),
            measurements=("measurement", "nunique"),
            meter_measurement_pairs=("measurement", "size"),
            csv_rows=("csv_rows", "sum"),
            inserted_rows=("inserted_rows", "sum"),
            null_value_rows=("null_value_rows", "sum"),
        )
        .reset_index()
    )
else:
    coverage_summary = coverage_df

show_table(coverage_summary, "coverage_summary.csv")

In [ ]:
# C08. Measurement 사전 결합 preview
measurement_dict_df = query_df("""
SELECT measurement, unit, measurement_family, description
FROM ems.full_measurement_definition
ORDER BY measurement
""")

show_table(measurement_dict_df, "measurement_dictionary.csv")

if "error_type" not in coverage_df.columns and "error_type" not in measurement_dict_df.columns:
    missing_definition_df = (
        coverage_df[["measurement"]]
        .drop_duplicates()
        .merge(measurement_dict_df[["measurement"]], on="measurement", how="left", indicator=True)
        .query("_merge == 'left_only'")
        .drop(columns="_merge")
        .sort_values("measurement")
    )
else:
    missing_definition_df = pd.DataFrame()

show_table(missing_definition_df, "missing_measurement_definition.csv")

In [ ]:
# C09. EDA 목적별 대표 meter 시간 격자 gap smoke check
team_correlation_meters = {
    "H1.Z10", "H1.Z13", "H1.Z16", "H1.Z20", "H2.T.Z33", "H2.Z35",
    "H2.Z64", "H2.Z68", "H2.ZE64", "H4.Z50", "V.Z84",
}

representative_meter_basis = pd.DataFrame([
    {"meter_urn": "H1.Z16", "domain": "electricity", "role": "consumption", "group": "central_cooling", "selection_axis": "central cooling electric load", "selection_reason": "중앙 냉각기 전력 부하의 핵심 계량기. 열 계통 V.K21, 기상 Ta/Igc와 coupling EDA 연결"},
    {"meter_urn": "H2.Z64", "domain": "electricity", "role": "consumption", "group": "server_power", "selection_axis": "server electric load", "selection_reason": "서버 전원 계통 대표. 상시부하와 냉방/서버 열 계통 비교에 필요"},
    {"meter_urn": "H1.Z20", "domain": "electricity", "role": "production", "group": "chp", "selection_axis": "CHP production sign", "selection_reason": "CHP 발전 부호 규약과 W/W_in/W_out 방향성 확인에 필요"},
    {"meter_urn": "V.Z84", "domain": "electricity", "role": "production", "group": "pv", "selection_axis": "PV-weather coupling", "selection_reason": "PV 발전량과 일사량·기온 coupling EDA의 기준 계량기"},
    {"meter_urn": "V.K21", "domain": "thermal", "role": "thermal_flow", "group": "cooling_thermal", "selection_axis": "central cooling thermal output", "selection_reason": "중앙 냉각 열량 계통. H1.Z16 전력과 전기-열 coupling 확인"},
    {"meter_urn": "H1.K16", "domain": "thermal", "role": "thermal_flow", "group": "server_thermal", "selection_axis": "server thermal load", "selection_reason": "서버 열 계통. H2.Z64 전력과 서버 부하 coupling 확인"},
    {"meter_urn": "H1.W11", "domain": "thermal", "role": "thermal_flow", "group": "heat_generation", "selection_axis": "heat generation", "selection_reason": "열 생산 계통. CHP 전력 생산과 열 생산 연결 확인"},
    {"meter_urn": "WeatherStation.Weather", "domain": "weather", "role": "weather", "group": "weather_station", "selection_axis": "external driver", "selection_reason": "냉방, PV, 서버 부하의 외생 변수 기준"},
])
representative_meter_basis["in_team_correlation_set"] = representative_meter_basis["meter_urn"].isin(team_correlation_meters)
show_table(representative_meter_basis, "representative_meter_selection_basis.csv")

representative_meters = representative_meter_basis["meter_urn"].tolist()
representative_measurements = ["P", "PF", "W", "W_in", "W_out", "Tdiff", "Trl", "Tvl", "V", "qv", "Igc", "Ta", "Ah"]


def gap_query(relation: str, expected_interval: str) -> pd.DataFrame:
    return query_df(f"""
    WITH base AS (
        SELECT
            ts,
            meter_urn,
            measurement,
            lag(ts) OVER (PARTITION BY meter_urn, measurement ORDER BY ts) AS prev_ts
        FROM ems.{relation}
        WHERE meter_urn = ANY(%s)
          AND measurement = ANY(%s)
    ), gaps AS (
        SELECT
            meter_urn,
            measurement,
            count(*) AS rows,
            min(ts) AS min_ts,
            max(ts) AS max_ts,
            sum(CASE WHEN prev_ts IS NOT NULL AND ts - prev_ts > interval '{expected_interval}' THEN 1 ELSE 0 END) AS gap_count,
            max(ts - prev_ts) AS max_observed_step
        FROM base
        GROUP BY meter_urn, measurement
    )
    SELECT '{relation}' AS relation, *
    FROM gaps
    ORDER BY meter_urn, measurement
    """, params=(representative_meters, representative_measurements), timeout_s=30)


gap_1h_df = gap_query("cr_measurement_1h", "1 hour")
gap_15min_df = gap_query("cr_measurement_15min", "15 minutes")
gap_df = pd.concat([gap_1h_df, gap_15min_df], ignore_index=True)

show_table(gap_df, "representative_time_grid_gaps.csv")

In [ ]:
# C10. EDA 목적별 대표 consumption/production meter 부호 분포 smoke check
sign_meters = ["H1.Z16", "H2.Z64", "H1.Z20", "V.Z84"]
sign_measurements = ["P", "W", "W_in", "W_out"]

sign_df = query_df("""
SELECT
    'cr_measurement_1h' AS relation,
    meter_urn,
    measurement,
    count(*) AS rows,
    sum(CASE WHEN value < 0 THEN 1 ELSE 0 END) AS negative_rows,
    sum(CASE WHEN value = 0 THEN 1 ELSE 0 END) AS zero_rows,
    sum(CASE WHEN value > 0 THEN 1 ELSE 0 END) AS positive_rows,
    min(value) AS min_value,
    percentile_cont(0.5) WITHIN GROUP (ORDER BY value) AS median_value,
    max(value) AS max_value
FROM ems.cr_measurement_1h
WHERE meter_urn = ANY(%s)
  AND measurement = ANY(%s)
GROUP BY meter_urn, measurement
ORDER BY meter_urn, measurement
""", params=(sign_meters, sign_measurements), timeout_s=30)

show_table(sign_df, "representative_sign_distribution.csv")

In [ ]:
# C11. 요약 시각화
if "error_type" not in source_status_df.columns and not source_status_df.empty:
    plot_df = source_status_df.copy()
    plot_df["scope"] = plot_df["processing_level"].astype(str) + " / " + plot_df["resolution_code"].astype(str)
    fig, ax = plt.subplots(figsize=(12, 5))
    sns.barplot(data=plot_df, x="scope", y="source_files", hue="status", ax=ax)
    ax.set_title("Source file status by processing level and resolution")
    ax.set_xlabel("processing_level / resolution_code")
    ax.set_ylabel("source files")
    ax.tick_params(axis="x", rotation=45)
    fig.tight_layout()
    if SAVE_OUTPUTS:
        OUT_FIG.mkdir(parents=True, exist_ok=True)
        fig_path = OUT_FIG / "source_status_summary.png"
        fig.savefig(fig_path, dpi=150)
        print(f"saved={fig_path}")
    plt.show()

if "error_type" not in quality_summary_df.columns and not quality_summary_df.empty:
    rate_cols = [col for col in quality_summary_df.columns if col.endswith("_rate")]
    if rate_cols:
        rate_df = quality_summary_df.melt(
            id_vars=["processing_level", "resolution_code", "status"],
            value_vars=rate_cols,
            var_name="counter",
            value_name="rate",
        )
        rate_df["scope"] = rate_df["processing_level"].astype(str) + " / " + rate_df["resolution_code"].astype(str) + " / " + rate_df["status"].astype(str)
        fig, ax = plt.subplots(figsize=(12, 6))
        sns.barplot(data=rate_df, x="counter", y="rate", hue="scope", ax=ax)
        ax.set_title("Quality counter rate by source scope")
        ax.set_xlabel("quality counter")
        ax.set_ylabel("rate over csv_rows")
        ax.tick_params(axis="x", rotation=45)
        fig.tight_layout()
        if SAVE_OUTPUTS:
            OUT_FIG.mkdir(parents=True, exist_ok=True)
            fig_path = OUT_FIG / "quality_counter_rates.png"
            fig.savefig(fig_path, dpi=150)
            print(f"saved={fig_path}")
        plt.show()

## 12. 해석 메모 작성 기준

이 노트북 실행 후 해석은 다음 순서로 작성한다.

1. `relation_inventory.csv`에서 `cr_measurement_1min` 존재 여부를 확인하고, `1min` 행은 `full_source_file` metadata 기준으로 해석한다.
2. `source_status_summary.csv`에서 loaded 이외 status가 있는지 확인한다.
3. `quality_counter_rates.csv`에서 NULL, invalid, non-finite, duplicate counter가 높은 scope를 확인한다.
4. `coverage_summary.csv`에서 15min/1h mart의 meter·measurement coverage 차이를 확인한다.
5. `representative_meter_selection_basis.csv`에서 대표 smoke check meter가 어떤 EDA 축을 대표하는지 확인한다.
6. `representative_time_grid_gaps.csv`에서 대표 meter의 시간 격자 결손을 확인한다.
7. `representative_sign_distribution.csv`에서 consumption/production 부호 규약과 실제 분포가 맞는지 확인한다.

`SAVE_OUTPUTS = False` 상태에서는 표와 그림이 노트북 출력에 남는다. 검토 후 보관할 snapshot이 정해진 경우에만 `SAVE_OUTPUTS = True`로 재실행한다.

후속 노트북은 이 결과를 기준으로 `01_group_aggregate_eda.ipynb`에서 group aggregate를 구성한다.